# ORM
Map database rows to Python objects while keeping queries explicit.


In [ ]:
# ORM models map Python objects to database tables and columns.
from sqlalchemy import Integer, String, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    email: Mapped[str] = mapped_column(String, unique=True)

engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(engine)
# A session groups database work and owns the transaction boundary.
with Session(engine) as session:
    session.add(User(email="ada@example.com"))
    session.commit()
    user = session.scalar(select(User).where(User.email == "ada@example.com"))
    print(user.email)


## Polished version
Keep session ownership at the application boundary and queries inside a repository.


In [ ]:
# Keep query details inside a repository instead of business services.
class UserRepository:
    def __init__(self, session: Session) -> None:
        self.session = session

    def by_email(self, email: str) -> User | None:
        return self.session.scalar(select(User).where(User.email == email))

with Session(engine) as session:
    repository = UserRepository(session)
    found = repository.by_email("ada@example.com")
    print(found.email if found else None)
